## Cell 1: Setup

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile
MYDRIVE = '/content/drive/MyDrive'
WORK = f'{MYDRIVE}/gan_vs_diffusion_work'; os.makedirs(WORK, exist_ok=True)
CODE = '/content/gan_vs_diffusion'
DATA = '/content/bbbc_128'

if not os.path.exists(f'{CODE}/ddpm/train.py'):
    z = f'{MYDRIVE}/gan_vs_diffusion.zip'
    if not os.path.exists(z):                      # e.g. "gan_vs_diffusion (1).zip"
        found = sorted(glob.glob(f'{MYDRIVE}/gan_vs_diffusion*.zip'))
        if not found:
            raise RuntimeError('Put gan_vs_diffusion.zip directly in My Drive, then run again.')
        z = found[0]
    shutil.rmtree('/content/_code', ignore_errors=True)
    zipfile.ZipFile(z).extractall('/content/_code')
    src = glob.glob('/content/_code/**/ddpm/train.py', recursive=True)[0]
    shutil.copytree(os.path.dirname(os.path.dirname(src)), CODE, dirs_exist_ok=True)
os.chdir(CODE)
!pip install -q tifffile torch-fidelity ninja click psutil imageio scikit-image
!rm -rf third_party/stylegan3          # always start from a fresh copy of NVIDIA's code
!python stylegan/setup_stylegan3.py

# Compatibility fixes for NVIDIA's code on current Colab (safe to run more than once)
SG = f'{CODE}/third_party/stylegan3'
p = f'{SG}/torch_utils/misc.py'
s = open(p).read()
open(p, 'w').write(s.replace('super().__init__(dataset)', 'super().__init__()'))
FIX = """

# gan_vs_diffusion: fall back to the reference implementation
_gvd_init_orig = _init
_gvd_failed = False

def _init():
    global _gvd_failed
    if _gvd_failed:
        return False
    try:
        return _gvd_init_orig()
    except Exception as e:
        _gvd_failed = True
        print(f'Note: custom CUDA kernel unavailable ({e.__class__.__name__}); using the reference implementation.')
        return False
"""
for name in ['bias_act.py', 'upfirdn2d.py', 'filtered_lrelu.py']:
    p = f'{SG}/torch_utils/ops/{name}'
    s = open(p).read()
    if 'gan_vs_diffusion: fall back' not in s:
        open(p, 'w').write(s + FIX)
# check NVIDIA's code is complete before training
import py_compile
assert 'def profiled_function' in open(f'{SG}/torch_utils/misc.py').read(), 'misc.py damaged'
for rel in ['torch_utils/misc.py', 'torch_utils/ops/bias_act.py', 'torch_utils/ops/upfirdn2d.py',
            'torch_utils/ops/filtered_lrelu.py', 'dnnlib/util.py', 'train.py']:
    py_compile.compile(f'{SG}/{rel}', doraise=True)
print('Setup done:', CODE)

## Cell 2: Unpacking prepared data from Drive

In [ ]:
if not os.path.exists(f'{DATA}/manifest.json'):
    zp = f'{WORK}/bbbc_128.zip'
    if not os.path.exists(zp):
        raise RuntimeError('bbbc_128.zip not found in gan_vs_diffusion_work - upload it from your Mac, then run again.')
    !unzip -oq {zp} -d /content
!ls {DATA} {DATA}/subsets/seed0

## Cell 3: Building BBBC038 test set

In [ ]:
!mkdir -p /content/datasets && cd /content/datasets && wget -q -nc https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip && mkdir -p stage1_train && unzip -oq stage1_train.zip -d stage1_train
!python data/make_bbbc038_test.py --bbbc038 /content/datasets/stage1_train --data-root {DATA} --out /content/bbbc038_test
if _exit_code != 0:
    raise RuntimeError('Building the test set failed - see the error above.')

## Cell 3b: Updated table code

In [ ]:
%%writefile /content/gan_vs_diffusion/eval/final_table.py
#!/usr/bin/env python3

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "eval"))
from metrics import compute  # noqa: E402


def first_session_snapshot(run):
    """The last snapshot of the first training session = the original 300-kimg model."""
    for sub in sorted(d for d in run.glob("*") if d.is_dir() and not d.name.startswith("_")):
        snaps = sorted(sub.glob("network-snapshot-*.pkl"), key=lambda p: int(p.stem.split("-")[-1]))
        if snaps and int(snaps[-1].stem.split("-")[-1]) > 0:
            return snaps[-1]
    return None


def last_snapshot(run):
    """The last snapshot of the last training session = the model at the end of all training (600 kimg)."""
    last = None
    for sub in sorted(d for d in run.glob("*") if d.is_dir() and not d.name.startswith("_")):
        snaps = sorted(sub.glob("network-snapshot-*.pkl"), key=lambda p: int(p.stem.split("-")[-1]))
        if snaps and int(snaps[-1].stem.split("-")[-1]) > 0:
            last = snaps[-1]
    return last


def generate_stylegan(pkl, out, n):
    import torch
    sys.path.insert(0, str(ROOT / "stylegan"))
    from generate import generate, load_generator
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"generating {n} images from {pkl.parent.name}/{pkl.name}", flush=True)
    generate(load_generator(pkl, dev), n, out, batch=32, seed=123, device=dev)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work", required=True)
    ap.add_argument("--test039", required=True)
    ap.add_argument("--test038", required=True, help="output folder of data/make_bbbc038_test.py")
    ap.add_argument("--n-gen", type=int, default=300)
    ap.add_argument("--sizes", nargs="+", type=int, default=[5, 25, 100])
    args = ap.parse_args()
    work = Path(args.work).expanduser()
    t38 = Path(args.test038).expanduser()
    out_dir = work / "results"; out_dir.mkdir(parents=True, exist_ok=True)
    cache = out_dir / "final_table_cache.json"          # resume-safe: finished rows are kept
    rows = json.loads(cache.read_text()) if cache.exists() else {}

    jobs = []
    for n in args.sizes:                                   # StyleGAN at 300 kimg and after continuing
        run = work / "runs" / f"stylegan2ada/scratch-n{n}-seed0"
        first = first_session_snapshot(run)
        if first is not None:
            d = Path(f"/tmp/gvd_sg300_n{n}") if Path("/content").exists() else run / "_sg300_samples"
            if len(list(d.glob("gen_*.png"))) < args.n_gen:
                generate_stylegan(first, d, args.n_gen)
            jobs.append((f"StyleGAN2-ADA, 300 kimg | n={n}", d, args.n_gen))
        last = last_snapshot(run)
        fm = run / "final_metrics.json"
        if fm.exists():
            print(f"StyleGAN n={n}: snapshot chosen by KID on BBBC039 validation = {json.loads(fm.read_text())['snapshot']}")
        if last is not None and last != first:              # training was continued
            d = Path(f"/tmp/gvd_sg600_n{n}") if Path("/content").exists() else run / "_sg600_samples"
            if len(list(d.glob("gen_*.png"))) < args.n_gen:
                generate_stylegan(last, d, args.n_gen)
            jobs.append((f"StyleGAN2-ADA, 600 kimg | n={n}", d, args.n_gen))
        elif first is None:
            print(f"skip StyleGAN n={n}: no snapshots found")
        else:
            print(f"StyleGAN n={n}: no continued training found (only one session with snapshots)")
    for n in args.sizes:
        d = work / "runs" / f"ddpm/n{n}-seed0" / "final_samples"
        if d.exists() and len(list(d.glob("gen_*.png"))) >= args.n_gen:
            jobs.append((f"DDPM | n={n}", d, args.n_gen))
        else:
            print(f"skip DDPM n={n}: final_samples missing (run its evaluation first)")
    jobs.append(("Real BBBC038 half A (reference)", t38 / "half_a", None))
    for name, gen, ng in jobs:
        for test_name, test_dir in [("BBBC039 test", args.test039),
                                    ("BBBC038 unseen", t38 / ("half_b" if "half A" in name else "images"))]:
            key = f"{name} || {test_name}"
            if key in rows:
                continue
            try:                                           # precision/recall (VGG) can fail to download
                m = compute(gen, test_dir, prc=True, max_gen=ng)
            except Exception as e:  # noqa: BLE001
                print(f"precision/recall failed ({e.__class__.__name__}); FID/KID only", flush=True)
                m = compute(gen, test_dir, max_gen=ng)
            rows[key] = {k: m[k] for k in ("fid", "kid", "kid_std", "n_gen", "precision", "recall") if k in m}
            cache.write_text(json.dumps(rows, indent=2))
            print(f"{key:60s} KID={m['kid']:.4f}  FID={m['fid']:.1f}  "
                  f"P={m.get('precision', float('nan')):.3f}  R={m.get('recall', float('nan')):.3f}", flush=True)

    names = [j[0] for j in jobs]
    lines = ["| Model | Training images | KID ↓ 039 test | FID ↓ 039 test | Prec ↑ 039 | Rec ↑ 039 "
             "| KID ↓ 038 unseen | FID ↓ 038 unseen | Prec ↑ 038 | Rec ↑ 038 |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for name in names:
        a, b = rows.get(f"{name} || BBBC039 test"), rows.get(f"{name} || BBBC038 unseen")
        model, size = (name.split(" | ") + [""])[:2] if " | " in name else (name, "–")
        fmt = lambda r, k, p: f"{r[k]:.{p}f}" if r else "–"  # noqa: E731
        kid = lambda r: f"{r['kid']:.3f} ± {r['kid_std']:.3f}" if r else "–"  # noqa: E731
        pr = lambda r, k: f"{r[k]:.3f}" if r and k in r else "–"  # noqa: E731
        lines.append(f"| {model} | {size.replace('n=', '')} | {kid(a)} | {fmt(a, 'fid', 1)} | {pr(a, 'precision')} | "
                     f"{pr(a, 'recall')} | {kid(b)} | {fmt(b, 'fid', 1)} | {pr(b, 'precision')} | {pr(b, 'recall')} |")
    lines += ["", f"Generated images per model: {args.n_gen}. The 'Real BBBC038 half A' row compares two disjoint "
              "halves of real unseen BBBC038 patches (in-domain floor) and real BBBC038 vs real BBBC039 "
              "(dataset-gap floor)."]
    table = "\n".join(lines)
    (out_dir / "final_table.md").write_text(table + "\n")
    print("\n" + table + f"\n\nSaved: {out_dir / 'final_table.md'}")


if __name__ == "__main__":
    main()


## Cell 4: Compute the final table

In [ ]:
!python eval/final_table.py --work {WORK} --test039 {DATA}/bbbc039/test/images --test038 /content/bbbc038_test
if _exit_code != 0:
    raise RuntimeError('Final table failed - see the error above.')

## Switch the GPU off

In [ ]:
from google.colab import runtime
runtime.unassign()